# Week 8 Lab 1 - Exact gas dynamics before machine learning

<!-- MIE690A article-aligned validation v4 -->

<!-- FLOWMLLAB_COLAB_LAUNCH_V1 -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ehsan-Roohi/FlowMLLab/blob/main/notebooks/week08/W8_Lab1_Exact_Gas_Dynamics_Student.ipynb)

**Runtime:** CPU, normally under 3 minutes. **Required background:** Mach number,
perfect-gas relations, and basic Python arrays.

### Central question

Before training a neural surrogate, can you identify the exact relation, its
physical domain, every solution branch, and the numerical operation that the
network is supposed to replace?

### Learning outcomes

By the end of this lab, you should be able to:

1. compute Rayleigh and Fanno reference ratios and explain choking from either side of $M=1$;
2. recover weak and strong oblique-shock roots without mixing the branches;
3. locate a normal shock inside a converging-diverging nozzle with a bounded root solve;
4. solve the shock-tube pressure compatibility equation and verify its residual;
5. classify a task as closed-form evaluation, bracketed inversion, ODE integration, or CFD; and
6. define what must remain exact when a learned approximation is introduced.


## Scientific-use contract

- The gas is calorically perfect unless a cell explicitly changes $\gamma$.
- All inverse problems are evaluated only on a declared physical branch and domain.
- Exact relations and bracketed roots are the references; a neural prediction is never its own validation target.
- A converged nonlinear solver is not automatically a physically admissible solution.
- The nine chapter notebooks in `Introduction-to-Compressible-Flows` remain the detailed classical source. This lab is a compact bridge into FlowMLLab.
- The retained formulas are synchronized to `GasDynamicsSciML` commit
  `374431a1033138f56e2752bf8bbf9b75a454d80c`.
- The teaching interpretation follows the author-supplied revised manuscript
  *Physics-Guided Neural Surrogates for Canonical Compressible Thermal-Fluid
  Relations* (`AITF-D-26-00044R1`). A revision identifier is not treated as
  evidence of editorial acceptance.


In [ ]:
# FLOWMLLAB_COLAB_BOOTSTRAP_V1
from pathlib import Path as _FlowMLLabPath
import os as _flowmllab_os
import subprocess as _flowmllab_subprocess
import sys as _flowmllab_sys

if "google.colab" in _flowmllab_sys.modules or _flowmllab_os.environ.get("COLAB_RELEASE_TAG"):
    _flowmllab_root = _FlowMLLabPath("/content/FlowMLLab")
    if not (_flowmllab_root / ".git").is_dir():
        _flowmllab_subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Ehsan-Roohi/FlowMLLab.git", str(_flowmllab_root)],
            check=True,
        )
    _flowmllab_subprocess.run(
        [_flowmllab_sys.executable, "-m", "pip", "install", "-q", "-e", str(_flowmllab_root)],
        check=True,
    )
    _flowmllab_os.chdir(_flowmllab_root / "notebooks/week08")

from pathlib import Path
import json
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print

REPO_ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "results/gas_dynamics_week8").is_dir()
)
if str(REPO_ROOT) not in _flowmllab_sys.path:
    _flowmllab_sys.path.insert(0, str(REPO_ROOT))
RESULTS = REPO_ROOT / "results/gas_dynamics_week8"
plt.rcParams.update({"font.size": 11, "axes.labelsize": 12, "legend.fontsize": 9})
print("Python:", platform.python_version())
print("FlowMLLab root:", REPO_ROOT)


In [ ]:
from flowmllab.gas_dynamics import (
    GAMMA, area_mach, fanno_inverse_friction_length, fanno_ratios,
    mach_from_area, nozzle_back_pressure, nozzle_shock_area,
    oblique_beta, oblique_detachment, oblique_theta,
    rayleigh_inverse_t0, rayleigh_ratios,
    shock_tube_pressure_ratio, shock_tube_residual_general,
)

print("gamma =", GAMMA)


## 1. Branches are part of the problem definition

Rayleigh flow models heat transfer in a constant-area frictionless duct. Fanno
flow models adiabatic flow with wall friction. In both models, the sonic state
is the reference state and the same normalized target can correspond to a
subsonic and a supersonic Mach number.

Predict first:

1. Does heating drive both branches toward or away from $M=1$?
2. Can a scalar target such as $T_0/T_0^*$ uniquely determine Mach number?
3. Why must `branch` be an input to an inverse solver or learned model?


In [ ]:
m_sub = np.linspace(0.15, 0.995, 260)
m_sup = np.linspace(1.005, 3.0, 300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
for mach, color, label in ((m_sub, "#2F75B5", "subsonic"),
                           (m_sup, "#C44536", "supersonic")):
    axes[0].plot(mach, rayleigh_ratios(mach)[:, 4], color=color, lw=2.3, label=label)
    axes[1].plot(mach, fanno_ratios(mach)[:, 4], color=color, lw=2.3, label=label)
for axis in axes:
    axis.axvline(1.0, color="#17365D", ls="--")
    axis.grid(alpha=0.3)
    axis.set_xlabel("Mach number")
axes[0].set(title="Rayleigh", ylabel=r"$T_0/T_0^*$")
axes[1].set(title="Fanno", ylabel=r"$4fL^*/D$", ylim=(0, 4))
axes[0].legend(frameon=False)
plt.show()

target_t0 = 0.82
rayleigh_roots = {
    branch: rayleigh_inverse_t0(target_t0, branch)
    for branch in ("subsonic", "supersonic")
}
target_friction = 0.18
fanno_roots = {
    branch: fanno_inverse_friction_length(target_friction, branch)
    for branch in ("subsonic", "supersonic")
}
display(pd.DataFrame([rayleigh_roots, fanno_roots], index=["Rayleigh", "Fanno"]))

assert rayleigh_roots["subsonic"] < 1.0 < rayleigh_roots["supersonic"]
assert fanno_roots["subsonic"] < 1.0 < fanno_roots["supersonic"]


### Checkpoint 1

Write one sentence explaining why a single-output regression
`target ratio -> Mach` is ill-posed if branch identity is hidden. Then change
each target above and verify by forward substitution that both returned roots
recover it.


In [ ]:
rayleigh_closure = {
    branch: float(rayleigh_ratios(mach)[4] - target_t0)
    for branch, mach in rayleigh_roots.items()
}
fanno_closure = {
    branch: float(fanno_ratios(mach)[4] - target_friction)
    for branch, mach in fanno_roots.items()
}
print("Rayleigh inverse closure:", rayleigh_closure)
print("Fanno inverse closure:", fanno_closure)
assert max(abs(value) for value in rayleigh_closure.values()) < 1e-9
assert max(abs(value) for value in fanno_closure.values()) < 1e-9


## 2. Oblique shocks: two attached roots and a detachment limit

For upstream Mach number $M_1$, shock angle $\beta$, and flow deflection
$\theta$, the theta-beta-M relation is

$$
\tan\theta = 2\cot\beta\,
\frac{M_1^2\sin^2\beta-1}{M_1^2(\gamma+\cos 2\beta)+2}.
$$

Below the maximum turning angle there are weak and strong roots. At the maximum
they merge; above it, an attached oblique shock is impossible. This is a
topological feature of the solution manifold, not a training-data nuisance.


In [ ]:
mach_1 = 2.0
beta_peak, theta_max = oblique_detachment(mach_1)
theta_requested = np.radians(12.0)
beta_weak = oblique_beta(mach_1, theta_requested, "weak")
beta_strong = oblique_beta(mach_1, theta_requested, "strong")

beta_grid = np.linspace(np.arcsin(1 / mach_1) + 1e-4, np.pi / 2 - 1e-4, 500)
theta_grid = oblique_theta(mach_1, beta_grid)
plt.figure(figsize=(7.4, 4.4))
plt.plot(np.degrees(beta_grid), np.degrees(theta_grid), color="#2A9D8F", lw=2.4)
plt.scatter(np.degrees([beta_weak, beta_strong]), [12, 12],
            color=["#2F75B5", "#C44536"], s=65, label="weak / strong roots")
plt.scatter(np.degrees([beta_peak]), np.degrees([theta_max]),
            color="#E9A23B", s=65, label="detachment")
plt.xlabel(r"shock angle $\beta$ (deg)")
plt.ylabel(r"turn angle $\theta$ (deg)")
plt.grid(alpha=0.3)
plt.legend(frameon=False)
plt.show()

print(f"weak beta = {np.degrees(beta_weak):.3f} deg")
print(f"strong beta = {np.degrees(beta_strong):.3f} deg")
print(f"theta_max = {np.degrees(theta_max):.3f} deg")
assert beta_weak < beta_peak < beta_strong


## 3. Nozzle shock location: bounded inverse design

For a fixed exit-to-throat area ratio, an internal normal shock maps its area
location $A_s/A_t$ to a back-pressure ratio. The inverse must remain in
$1 < A_s/A_t < A_e/A_t$. A generic unconstrained regressor can violate that
geometry even when its mean error looks small.


In [ ]:
exit_area_ratio = 2.5
shock_area_grid = np.linspace(1.0001, exit_area_ratio - 0.0001, 240)
back_pressure_grid = np.array([
    nozzle_back_pressure(exit_area_ratio, area) for area in shock_area_grid
])

target_index = 135
target_back_pressure = float(back_pressure_grid[target_index])
recovered_area = nozzle_shock_area(exit_area_ratio, target_back_pressure)

plt.figure(figsize=(7.4, 4.4))
plt.plot(shock_area_grid, back_pressure_grid, color="#E9A23B", lw=2.4)
plt.scatter([recovered_area], [target_back_pressure], color="#C44536", s=65)
plt.xlabel(r"shock area $A_s/A_t$")
plt.ylabel(r"back pressure $P_b/P_{01}$")
plt.grid(alpha=0.3)
plt.show()

print("target back pressure:", target_back_pressure)
print("recovered shock area:", recovered_area)
assert 1.0 < recovered_area < exit_area_ratio
assert abs(nozzle_back_pressure(exit_area_ratio, recovered_area) - target_back_pressure) < 1e-9


## 4. Shock tube: solve the compatibility equation, then inspect the residual

The star pressure behind the incident shock and ahead of the contact surface is
implicit. A bracketed scalar root is the transparent reference. The generalized
solver may also vary driver temperature, both heat-capacity ratios, and the gas
constant ratio; this is the five-input problem used later to study dimensional scaling.


In [ ]:
driver_ratios = np.geomspace(1.05, 50.0, 100)
star_pressures = np.array([shock_tube_pressure_ratio(value) for value in driver_ratios])
residuals = shock_tube_residual_general(
    star_pressures, driver_ratios, np.ones_like(driver_ratios)
)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.1), constrained_layout=True)
axes[0].loglog(driver_ratios, star_pressures, color="#2F75B5", lw=2.4)
axes[0].set(xlabel=r"$P_4/P_1$", ylabel=r"$P_2/P_1$", title="implicit pressure map")
axes[1].semilogx(driver_ratios, np.abs(residuals), color="#C44536", lw=2.2)
axes[1].set(xlabel=r"$P_4/P_1$", ylabel="absolute compatibility residual", title="closure check")
for axis in axes:
    axis.grid(alpha=0.3)
plt.show()

print("maximum compatibility residual:", np.max(np.abs(residuals)))
assert np.max(np.abs(residuals)) < 1e-9


## 5. Continue into the nine classical chapter notebooks

These are the canonical, CI-tested educational notebooks in
[`Introduction-to-Compressible-Flows`](https://github.com/Ehsan-Roohi/Introduction-to-Compressible-Flows)
at commit `fc14721ae80f48da63e55955c1caf096d8448f7b`:

| Topic | Numerical idea | Open in Colab |
|---|---|---|
| Rayleigh flow | sonic reference, heat-addition branches, bisection | [Chapter 4](https://colab.research.google.com/github/Ehsan-Roohi/Introduction-to-Compressible-Flows/blob/main/notebooks/chapter04/04_rayleigh_flow_ai_solver.ipynb) |
| Emanuel oblique shock | explicit weak/strong construction | [6.1](https://colab.research.google.com/github/Ehsan-Roohi/Introduction-to-Compressible-Flows/blob/main/notebooks/chapter06/06_01_emanuel_oblique_shock.ipynb) |
| Shock polar | velocity-space geometry | [6.2](https://colab.research.google.com/github/Ehsan-Roohi/Introduction-to-Compressible-Flows/blob/main/notebooks/chapter06/06_02_shock_polar.ipynb) |
| Shock collision and slip line | pressure compatibility | [6.3](https://colab.research.google.com/github/Ehsan-Roohi/Introduction-to-Compressible-Flows/blob/main/notebooks/chapter06/06_03_oblique_shock_collision_slip_line.ipynb) |
| Shock tube | incident shock, expansion, contact matching | [7.1](https://colab.research.google.com/github/Ehsan-Roohi/Introduction-to-Compressible-Flows/blob/main/notebooks/chapter07/07_01_shock_tube_pressure_solver.ipynb) |
| Interacting shocks | corrected nonlinear pressure-ratio equation | [7.2](https://colab.research.google.com/github/Ehsan-Roohi/Introduction-to-Compressible-Flows/blob/main/notebooks/chapter07/07_02_interacting_shock_pressure_ratios.ipynb) |
| C-D nozzle | normal-shock location | [Chapter 8](https://colab.research.google.com/github/Ehsan-Roohi/Introduction-to-Compressible-Flows/blob/main/notebooks/chapter08/08_normal_shock_location_cd_nozzle.ipynb) |
| Conical flow | Taylor-Maccoll RK4 integration | [10.1](https://colab.research.google.com/github/Ehsan-Roohi/Introduction-to-Compressible-Flows/blob/main/notebooks/chapter10/10_01_conical_flow_from_shock_angle.ipynb) |
| Cone sweep | ODE event detection and surface state | [10.2](https://colab.research.google.com/github/Ehsan-Roohi/Introduction-to-Compressible-Flows/blob/main/notebooks/chapter10/10_02_taylor_maccoll_cone_sweep.ipynb) |

### Solver classification

For each row, record whether the reference is an algebraic evaluation, a
branch-wise scalar root, an ODE initial-value/event problem, or a multidimensional
CFD calculation. That classification determines the proper baseline and validation.


## 6. Bridge to multidimensional compressible CFD

The separate
[`SU2-Diamond-Airfoil-Verification`](https://github.com/Ehsan-Roohi/SU2-Diamond-Airfoil-Verification)
repository advances from canonical relations to a Mach-3 diamond airfoil at
$\alpha=0,4,8$ degrees with Euler, laminar Navier-Stokes, and SST RANS configurations.

At the frozen Week-8 source commit, only the sharp-wall `euler_alpha0` case is a
**qualified teaching reference**. The other eight distributed cases remain
unverified. Therefore they appear here as a future verification project, not as
accepted CFD labels or ML training data.

A student must check residual reduction, force-window stability, physicality
warnings, symmetry, shock angle, and wall resolution before promoting any SU2
output into the FlowMLLab evidence chain.


## Exit ticket

Submit a one-page evidence card containing:

1. one inverse problem and its declared domain;
2. every valid branch or bound;
3. the exact/numerical reference operation;
4. one closure residual evaluated by forward substitution;
5. the strongest non-neural baseline you would test before an MLP; and
6. one result that would make you reject the learned model despite a small global error.

Proceed to Lab 2 only after your reference solver and branch contract are explicit.
